In [1]:
import os
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential,load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Layer, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

physical_devices = tf.config.experimental.list_physical_devices('GPU')
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

2024-06-23 13:31:02.474266: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-23 13:31:02.474398: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-23 13:31:02.599532: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
data = pd.read_csv('/kaggle/input/dataset/Impute_misvalues_hungyen.csv')
data.head()

,Date,Hour,Waterlevel
0,01/01/2008,0,47.0
1,01/01/2008,1,43.0
2,01/01/2008,2,40.0
3,01/01/2008,3,37.0
4,01/01/2008,4,34.0


In [3]:
data['Hour'] = data['Hour'].replace('#NUM!', np.nan)
data['Hour'] = pd.to_numeric(data['Hour'], errors='coerce')

# Convert 'Date' and 'Hour' to datetime
data['DateTime'] = pd.to_datetime(data['Date'] + ' ' + data['Hour'].fillna(0).astype(int).astype(str) + ':00')
data.set_index('DateTime', inplace=True)
data.drop(['Date', 'Hour'], axis=1, inplace=True)

# Standardize the 'Waterlevel' column
scaler = StandardScaler()
data['Waterlevel'] = scaler.fit_transform(data[['Waterlevel']])

In [4]:
time_steps = 18

def create_sequences(data, time_steps):
    sequences = []
    labels = []
    for i in range(len(data) - time_steps):
        seq = data[i:i + time_steps]
        label = data[i + time_steps]
        sequences.append(seq)
        labels.append(label)
    return np.array(sequences), np.array(labels)

In [5]:
x, y = create_sequences(data['Waterlevel'].values, time_steps)

x = x.reshape((x.shape[0], x.shape[1], 1))

train_size = int(len(x) * 0.8)
val_size = int(len(x) * 0.1)
test_size = len(x) - train_size - val_size

x_train, y_train = x[:train_size], y[:train_size]
x_val, y_val = x[train_size:train_size + val_size], y[train_size:train_size + val_size]
x_test, y_test = x[train_size + val_size:], y[train_size + val_size:]

print(f'x_train shape: {x_train.shape}, y_train shape: {y_train.shape}')
print(f'x_val shape: {x_val.shape}, y_val shape: {y_val.shape}')
print(f'x_test shape: {x_test.shape}, y_test shape: {y_test.shape}')

x_train shape: (51234, 18, 1), y_train shape: (51234,)
x_val shape: (6404, 18, 1), y_val shape: (6404,)
x_test shape: (6405, 18, 1), y_test shape: (6405,)


In [6]:
# class Attention(Layer):
#     def __init__(self, **kwargs):
#         super(Attention, self).__init__(**kwargs)

#     def build(self, input_shape):
#         self.W = self.add_weight(shape=(input_shape[-1], input_shape[-1]),
#                                  initializer='glorot_uniform',
#                                  trainable=True)
#         self.b = self.add_weight(shape=(input_shape[-1],),
#                                  initializer='zeros',
#                                  trainable=True)
#         self.u = self.add_weight(shape=(input_shape[-1],),
#                                  initializer='glorot_uniform',
#                                  trainable=True)
#         super(Attention, self).build(input_shape)

#     def call(self, x):
#         uit = tf.tensordot(x, self.W, axes=[2, 0]) + self.b
#         uit = tf.tanh(uit)
#         ait = tf.tensordot(uit, self.u, axes=[2, 0])
#         a = tf.nn.softmax(ait, axis=1)
#         a = tf.expand_dims(a, axis=-1)
#         output = x * a
#         return tf.reduce_sum(output, axis=1)

In [7]:
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight', shape=(input_shape[-1], input_shape[-1]),
                                 initializer='uniform', trainable=True)
        self.b = self.add_weight(name='attention_bias', shape=(input_shape[-1],),
                                 initializer='uniform', trainable=True)
        super(AttentionLayer, self).build(input_shape)

    def call(self, x):
        e = tf.nn.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        a = tf.nn.softmax(e, axis=1)
        output = x * a
        return tf.reduce_sum(output, axis=1)

In [8]:
model = Sequential([
    Input(shape=(time_steps, 1)), 
    Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
    Dropout(0.2),
    BatchNormalization(),
    Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'),
    Dropout(0.2),
    BatchNormalization(),
    Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
    Dropout(0.2),
    BatchNormalization(),
    Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'),
    Dropout(0.2),
    BatchNormalization(),
    AttentionLayer(),
    AttentionLayer(),
    Flatten(),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(1, activation='linear')
])

In [9]:
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model
history = model.fit(x_train, y_train, epochs=100, batch_size=10, validation_data=(x_val, y_val), callbacks=[early_stopping])

Epoch 1/100
  59/5124 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - loss: 0.6919 

I0000 00:00:1719149487.532668     105 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


5124/5124 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - loss: 0.0860 - val_loss: 0.0040
Epoch 2/100
5124/5124 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - loss: 0.0176 - val_loss: 0.0032
Epoch 3/100
5124/5124 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - loss: 0.0109 - val_loss: 0.0014
Epoch 4/100
5124/5124 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - loss: 0.0109 - val_loss: 0.0018
Epoch 5/100
5124/5124 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - loss: 0.0101 - val_loss: 0.0029
Epoch 6/100
5124/5124 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - loss: 0.0085 - val_loss: 0.0023
Epoch 7/100
5124/5124 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - loss: 0.0079 - val_loss: 0.0018
Epoch 8/100
5124/5124 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - loss: 0.0072 - val_loss: 0.0027
Epoch 9/100
5124/5124 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - loss: 0.0070 - val_loss: 0.0034
Epoch 10/100
5124/5124 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - loss: 0.0065 - val_loss: 0.0039
Epoch 11/100
5124/5124 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - loss: 0.0061 - val_loss: 0.0047
Epoch 12/100
5124/51

In [10]:
model.save('CNN_AM.h5')

In [31]:
import tensorflow as tf
from tensorflow.keras.utils import custom_object_scope
with custom_object_scope({'AttentionLayer': AttentionLayer}):
    model = tf.keras.models.load_model('CNN_AM.h5')

In [32]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 18, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 18, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 18, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 18, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 18, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 18, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 18, 64)         │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 18, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 18, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 18, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 18, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 18, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_layer                 │ (None, 128)            │        16,512 │
│ (AttentionLayer)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_layer_1               │ (None)                 │        16,512 │
│ (AttentionLayer)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 142,403 (556.27 KB)

 Trainable params: 141,633 (553.25 KB)

 Non-trainable params: 768 (3.00 KB)

 Optimizer params: 2 (12.00 B)

In [33]:
y_pred = model.predict(x_test)
print(y_pred)

201/201 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
[[ 0.6832789 ]
 [ 0.67032075]
 [ 0.6496206 ]
 ...
 [-0.21726061]
 [-0.26674843]
 [-0.32036144]]


In [ ]:
# model.save('CNN_AM.h5')
# model = models.load_model('CNN_AM.h5')

In [ ]:
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    squared_error = np.mean((y_true - y_pred)**2)

    r, _ = pearsonr(y_true, y_pred)
    
    T = len(y_true)
    max_x = max(y_true)
    min_x = min(y_true)
    
    sim_sum = 0
    for i in range(T):
        sim_sum += 1 / (1 + abs(y_pred[i] - y_true[i]) / (max_x - min_x))
    sim = sim_sum / T
    
    sd_y = np.std(y_pred)
    sd_x = np.std(y_test)
    fsd = 2 * np.abs(sd_y - sd_x) / (sd_y + sd_x)
    
    nse = 1 - (np.sum((y_pred - y_true) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2))
    
    return {
        'MAE': mae,
        'RMSE': rmse,
        'MSE' : squared_error,
        'R': r,
        'SIM': sim,
        'FSD': fsd,
        'NSE': nse
    }

In [ ]:
time = 77
hours = 120
pred = []

def forecast(model, hours, time):
    pred = []
    # current_window = x_val[-1,:].tolist()
    current_window = x_test[time,:].tolist()
    for i in range(hours):
        # Chuyển current_window thành numpy array khi gọi predict
        y_pred = model.predict(np.asarray([current_window]))[0]
        pred.append(float(y_pred[0]))
        current_window.pop(0)
        current_window.append(pred[-1])
    pred_array = np.array(pred).reshape(-1, 1)
    return pred_array

y_fc = forecast(model, hours, time)

In [ ]:
y_pred = model.predict(x_test[time:])

In [ ]:
y_test_original = scaler.inverse_transform(y_test)
y_pred_original = scaler.inverse_transform(y_pred)
y_fc_original = scaler.inverse_transform(y_fc)

In [ ]:
time_forecast = [3,  6, 12, 24, 36, 72, 120]
for hours in time_forecast:
    plt.figure(figsize=(20, 6))
    
    plt.plot(y_test_original[time : time + hours].flatten(), label='Actual', color='blue', marker='o')
    plt.plot(y_pred_original[: hours], label='Predicted', color='red', marker='x')
    plt.plot(y_fc_original[: hours].flatten(), label='Forecast', color='green', marker='x')
    plt.title('Comparison of Actual and Predicted Values')
    plt.xlabel('Hours')
    plt.ylabel('Waterlevel')
    plt.legend()

    plt.show()

In [ ]:
time_forecast = [3,  6, 12, 24, 36, 72, 120]
for hours in time_forecast:
    metrics = calculate_metrics(y_test_original[time : time + hours].flatten(), y_fc_original[:hours].flatten())
    for metric, value in metrics.items():
        print(f'Using model LSTM {hours} hours {metric}: {value}')

    print('\n')